# IMPORT AND SET

In [33]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
from matplotlib import gridspec
from matplotlib.ticker import AutoMinorLocator, LogLocator, AutoLocator
from matplotlib.animation import FuncAnimation, FFMpegWriter

plt.rcParams.update({
    'text.usetex' : True,
    "font.family": "serif",
    'font.size': 22,
    'axes.labelsize' : 'large',
    'lines.linewidth': 3,
    'figure.dpi': 100,
    'axes.linewidth' : 2,
    'xtick.direction': 'in',
    'xtick.major.size': 8,
    'xtick.minor.size': 4,
    'ytick.direction': 'in',
    'ytick.major.size': 8,
    'ytick.minor.size': 4,
    'xtick.major.width': 2,
    'xtick.minor.width': 1,
    'ytick.major.width': 2,
    'ytick.minor.width': 1,
    'xtick.major.pad' : 5,
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.top': True,
    'ytick.minor.right': True}
)


import h5py as h5
from scipy.optimize import fmin, curve_fit, minimize
from tqdm import tqdm


%matplotlib qt5
#%matplotlib inline

# ALL THE FUNCTIONS WE NEED

In [34]:
### Zonal average
def tilde(u):
    return u - np.mean(u, axis=-1)

### Isotropic spectrum
def Spectrum(F, kx, ky):
    kn = (kx**2 + ky**2)**0.5
    k = kx[:int(kx.shape[0]/2),0]
    dk = kx[1,0]
    S = np.zeros(len(k))
    for i in range(len(k)):
        ring = np.where((kn>=k[i]-dk/2)&(kn<k[i]+dk/2)&(ky>0))
        S[i] = np.sum(F[ring]) + np.sum(F[i,0])
    return S

### Smoothing functions for periodic zonal profile
def p_smooth(y):
    # smooth function flat at 0
    p = np.zeros_like(y)
    idp = y>0
    p[idp] = np.exp(-1/y[idp])
    return p

def jump(y):
    # smooth transition function
    return p_smooth(y) / (p_smooth(y) + p_smooth(1-y))

def smooth_gate(X, i1, i2, d_ix):
    # smooth gate function
    f = np.ones_like(X)
    il, ir = i1 -d_ix, i2 + d_ix
    idl = abs(X - (X[i1] + X[il]) /2) < (X[i1] - X[il]) /2  # points X[il] < X < X[ib1]
    idr = abs(X - (X[ir] + X[i2]) /2) < (X[ir] - X[i2]) /2 # points X[ib2] < X < X[ir]
    f[X <= X[il]] =0
    f[idl] = jump((X[idl] - X[il]) / (X[i1] - X[il]))
    f[idr] = jump((X[ir] - X[idr]) / (X[ir] - X[i2]))
    f[X >= X[ir]] =0
    return f
    
### Decompose and smooth the zonal profile in the buffer
def dec_prof(nr, X, ib1, ib2, im1, im2, d_ix):
    
    ### Get kappa and the linear profile
    kap = - (nr[ib2] - nr[ib1]) / (X[ib2] - X[ib1])
    nlin = -kap * (X - X[ib2]) + nr[ib2]

    ### Get the zonal profile from the radial profile
    nbar_raw = nr - nlin  
    
    ### Modify the zonal profile to make it periodic in the buffer
    n_off = np.mean(nbar_raw[np.r_[0, im1, im2, -1]]) #offset to shift the zonal profile with when multiplying by the smooth gate
    nbar_smth = (nbar_raw - n_off) * smooth_gate(X, im1, im2, d_ix) + n_off

    ### Remove the mean value from the zonal profile
    nm = np.mean(nbar_smth)
    nbar_smth -= nm
    return nbar_smth, kap, nm

def oneover(x):
    if(np.isscalar(x)):
        res=(1/x if x!=0 else 0)
    else:
        res=np.zeros_like(x)
        inds=np.nonzero(x)
        res[inds]=1/x[inds]
    return res

### Solve relation of dispersion of Hasegawa-Wakatani
def Wks(kx, ky, C, kap, nu=0, D=0, dxom=0, dxn=0, u=0):
    ksqr = kx**2+ky**2
    iksqr = oneover(ksqr)
    Ak = 0.5*((D+nu)*kx**2+(D+nu)*ky**2+C+C*iksqr+1j*dxom*ky*iksqr)
    Bk = 0.5*((D-nu)*kx**2+(D-nu)*ky**2+C-C*iksqr-1j*dxom*ky*iksqr)
    Gk = C**2*oneover(ksqr)+Bk.real**2-Bk.imag**2
    Hk = np.sqrt(Gk**2+(C*(kap-dxn)*ky*oneover(ksqr)-2*Bk.real*Bk.imag)**2)
    Wkp = np.sign(ky)*np.sqrt((Hk-Gk)/2)+1j*np.sqrt((Hk+Gk)/2)
    Wkm = -np.sign(ky)*np.sqrt((Hk-Gk)/2)-1j*np.sqrt((Hk+Gk)/2)
    om_p = Wkp+u*ky-1j*Ak
    om_m = Wkm+u*ky-1j*Ak
    return om_p, om_m

### Plot hatches in rectangles
def plot_hatches(ax, x0, y0, w, h, alpha=0.5):
    ax.add_patch(Rectangle((x0, y0), w, h, ec='k', fc='None',  hatch='\\\\', lw = 2, zorder=4, alpha=alpha))

## LOAD A SIMULATION

In [36]:
flname = 'outC1.0_1024x1024.h5'

with h5.File(flname, 'r', libver='latest', swmr=True) as fl:

    #Construct the real space grid
    Lx, Ly = fl['params/Lx'][()], fl['params/Ly'][()]
    Npx, Npy = fl['params/Npx'][()], fl['params/Npy'][()]
    Nx, Ny, Nxh = int(Npx/3)*2, int(Npy/3)*2, int(Npx/3)
    X, Y = np.arange(0, Lx, Lx/Nx), np.arange(0, Ly, Ly/Ny) 
    x, y = np.meshgrid(X, Y, indexing='ij')

    Nt = fl['fields/t'][()].size
    
    #Construct Fourier grid
    dkx, dky = 2*np.pi/Lx, 2*np.pi/Ly
    kx, ky = np.r_[np.arange(0,int(Nx/2)+1)*dkx, np.arange(-int(Nx/2)+1, 0)*dkx],  np.arange(0, int(Ny/2)+1)*dky
    kx, ky = np.meshgrid(kx, ky, indexing='ij')
    ksqr=kx**2+ky**2

    # Linear parameters
    C, nu, D = fl['params/C'][()], fl['params/nu'][()], fl['params/D'][()]
    
    # Buffer indices
    ib1, ib2, d_ixb, im1, im2, d_ixm = fl['buffer/indices'][()]
    mupen = fl['buffer/mupen'][()]
    sigS = fl['buffer/sigS'][()]

    # Compute linear properties using kap=1.0 (close to kap of initial profile), which are used to set timesteps, box size and dissipation
    kap = 1.0
    ky0 = fmin(lambda x: -np.imag(Wks(0, x, C, kap,nu=0, D=0) [0]) if x>=0 else 0, 1.25, disp=False)[0] #most unstable wavenumber
    gam = np.imag(Wks(0, ky0, C, kap,nu=0, D=0)[0]) #most unstable growth rate

    alpha, X0, sig_n = fl['source/alpha'][()], fl['source/X0'][()], fl['source/sig_n'][()]
    
    def S_n(X, X0, t, alpha, sig_n):
        return alpha * np.exp(-(X - X0) ** 2 / (2 * sig_n**2)) 

## Snapshots

In [37]:
it = -1

with h5.File(flname, 'r', libver='latest', swmr=True) as fl:
    uk = fl['fields/uk'][it]
    t_it = fl['fields/t'][it]


with h5.File(flname, 'r', libver='latest', swmr=True) as fl:    
    ur = fl['fields/ur'][it]
    nr = fl['fields/nr'][it]
    
    
# Computing the Fourier transform of om and n
om = np.fft.irfft2(-ksqr*uk[0,], norm='forward') 
n = np.fft.irfft2(uk[1,], norm='forward')

# Getting zonal velocity ubar and density profiles
nbar, kap, nm = dec_prof(nr, X, ib1, ib2, im1, im2, d_ixm)
nlin = -kap * (X - X[ib2]) + nr[ib2]


vmin_u, vmax_u = -1.5*np.max(abs(ur)), 1.5*np.max(abs(ur))
vmin_n, vmax_n = np.min(np.r_[nr, nbar + np.mean(nlin) + nm])-10, np.max(np.r_[nr, nbar + np.mean(nlin) + nm])+10

### Plot ###
fig=plt.figure(figsize=(15,8))
gs = gridspec.GridSpec(3, 2, height_ratios=[0.05, 20, 0.5], hspace=0.5, wspace=0.4)

# Create subplotsof
ax = [fig.add_subplot(gs[1, i]) for i in range(2)]
axi = [ax[i].twinx() for i in range(2)]
axi0, axi1 = axi[0], axi[1]


# Vorticity snapshot
ax[0].set_xlabel('$x$')
ax[0].set_ylabel('$y$')
ax[0].set_title(r'$\Omega(x,y,t)$')
qd_om =ax[0].pcolormesh(x, y, om, cmap='seismic', vmin=-0.6*np.max(abs(om)), vmax=0.6*np.max(abs(om)), rasterized=True)
ax[0].set_xlim([x[0,0],x[-1,0]])
ax[0].set_ylim([y[0,0],y[0,-1]])

# Zonal velocity profile
axi0.plot(x[:,0], ur, lw=6, color='w', alpha=0.9, zorder=2)
axi0.plot(x[:,0], ur, lw=4, color='k', alpha=0.9, zorder=2)
axi0.set_ylabel(r'$\overline{v}_y(x)$')
axi0.set_ylim([vmin_u, vmax_u])

# Density snapshot
ax[1].set_xlabel('$x$')
ax[1].set_ylabel('$y$')
ax[1].set_title(r'$n(x,y,t)$')
qd_n = ax[1].pcolormesh(x, y, n, cmap='seismic', vmin=-0.6*np.max(abs(n)), vmax=0.6*np.max(abs(n)), rasterized=True)

ax[1].set_xlim([x[0,0],x[-1,0]])
ax[1].set_ylim([y[0,0],y[0,-1]])

# Density complete, linear and zonal prifles
axi1.plot(x[:,0], nr, lw=6, color='w', alpha=0.9, zorder=2)
axi1.plot(x[:,0], nr, lw=4, color='k', alpha=0.9, zorder=2)

axi1.set_ylabel(r'$n_r(x)$')
axi1.set_ylim([0, vmax_n])

# Buffer profiles
ax[0].axvspan(0, X[ib1], alpha=0.5, color='grey', zorder=2)
ax[0].axvspan(X[ib2], X[-1], alpha=0.5, color='grey', zorder=3)
ax[1].axvspan(0, X[ib1], alpha=0.5, color='grey', zorder=3)
ax[1].axvspan(X[ib2], X[-1], alpha=0.5, color='grey', zorder=3)
plot_hatches(axi0, 0, axi0.get_ylim()[0], X[ib1], axi0.get_ylim()[1] - axi0.get_ylim()[0])
plot_hatches(axi0, X[ib2], axi0.get_ylim()[0], Lx - X[ib2], axi0.get_ylim()[1] - axi0.get_ylim()[0])
plot_hatches(axi1, 0, axi1.get_ylim()[0], X[ib1], axi1.get_ylim()[1] - axi1.get_ylim()[0])
plot_hatches(axi1, X[ib2], axi1.get_ylim()[0], Lx - X[ib2], axi1.get_ylim()[1] - axi1.get_ylim()[0])

cbar_ax1 = fig.add_subplot(gs[2, 0])
cbar_ax2 = fig.add_subplot(gs[2, 1])
cb1=fig.colorbar(qd_om, cax=cbar_ax1, orientation='horizontal')
cb2=fig.colorbar(qd_n, cax=cbar_ax2, orientation='horizontal')
cb1.set_label(r'$\Omega(x,y)$', labelpad=-10)
cb2.set_label(r'$n(x,y)$', labelpad=-10)

fig.suptitle(f'$t = {np.round(t_it,2)}$, $\\kappa = {np.round(kap,4)}$, $C/\\kappa = {np.round(C/kap,5)}$')

Text(0.5, 0.98, '$t = 40.0$, $\\kappa = 1.1957$, $C/\\kappa = 0.83632$')

## LOAD 0D and 1D

In [38]:
dt =1
Nt =-1
with h5.File(flname, 'r', libver='latest', swmr=True) as fl:
    t=fl['fields/profiles/t'][:Nt:dt]
    ur = np.zeros((t.size, X.size))
    nr = np.zeros((t.size, X.size))  
    K = np.zeros(t.size)
    Kbar = np.zeros(t.size)
    G_x = np.zeros((t.size, X.size)) 
    
    nbar, kap_t, nm_t = np.zeros((t.size, X.size)), np.zeros(t.size), np.zeros(t.size)#, np.zeros_like(nr)
    for it in tqdm(range(0, t.size)):
        ur[it] = fl['fields/profiles/ur'][int(it*dt)]
        nr[it] = fl['fields/profiles/nr'][int(it*dt)]    
        K[it] = fl['fields/energies/K'][int(it*dt)]
        Kbar[it] = fl['fields/energies/Kbar'][int(it*dt)]
        G_x[it] = fl['fields/fluxes/Gam'][int(it*dt)]
        nb, kn, nm = dec_prof(nr[it], X, ib1, ib2, im1, im2, d_ixm)
        nbar[it], kap_t[it], nm_t[it] = nb, kn, nm
    

zmf = Kbar/ K

100%|█████████████████████████████████████████████████████████████| 477/477 [00:00<00:00, 3989.77it/s]


## Energy, flux, density

In [39]:
fig, ax = plt.subplots(1, 3, figsize=(16,3.5), constrained_layout=True)

ax[0].plot(t, K, color='k', label=r'$\mathcal{K}$')
ax[0].plot(t, (K-Kbar), ls='dashed', color='green', label=r'$\mathcal{K}_{turb}$')
ax[0].plot(t, Kbar, color='r', ls=(0, (4,2,1,2)), label=r'$\mathcal{K}_{ZF}$')
ax[0].set_xlim([t[0]-30, t[-1]])
ax[0].set_ylim([1e-7,6e1])
ax[0].set_yscale('log')
ax[0].legend(fontsize=22, framealpha=0, ncol=2, loc='lower right')
ax[0].set_xlabel(r'$t$')
ax[0].set_ylabel(r'$\mathcal{K}$')

ax[0].xaxis.set_minor_locator(AutoMinorLocator(5))

# ax[0].grid()

ax[1].plot(t, nr[:, ib1], label=r'$n_r(x_{b1},t)$')

ax[2].semilogy(t, np.mean(G_x[:,ib1:ib2+1], axis=-1), label=r'$\langle \Gamma_n \rangle (t)$')

l,h = ax[1].get_legend_handles_labels()
li,hi = axi1.get_legend_handles_labels()


ax[1].set_xlabel(r'$t$')
ax[1].set_ylabel(r'$n_r(x_{b1},t)$', labelpad=10)
ax[2].set_ylabel(r'$\langle \Gamma_n \rangle (t)$')

ax[1].xaxis.set_minor_locator(AutoMinorLocator(5))
ax[1].yaxis.set_minor_locator(AutoMinorLocator(5))

## Hovmöller

In [41]:
dtplot = 10

tm, xm = np.meshgrid(t, X, indexing='ij') #run with tred

fig=plt.figure(figsize=(15,5))
gs = gridspec.GridSpec(2, 2, height_ratios=[10, 0.5], hspace=0.5, wspace=0.2)
ax = [fig.add_subplot(gs[0, i]) for i in range(2)]

ax[0].sharex(ax[1])
ax[0].sharey(ax[1])

ax[0].set_xlabel('$x$')
ax[0].set_ylabel('$t$')
qd_vy = ax[0].pcolormesh(xm[::dtplot,:], tm[::dtplot,:], ur[::dtplot,:]-np.mean(ur[-20000:], axis=0)[None,:]*0, vmin=-0.5*np.max(abs(ur)), vmax=0.5*np.max(abs(ur)), cmap='seismic', rasterized=True)
plot_hatches(ax[0], 0, 0, X[ib1], t[-1])
plot_hatches(ax[0], X[ib2+1], 0, Lx - X[ib2], t[-1])
ax[0].axvspan(0, X[ib1], alpha=0.4, color='grey')
ax[0].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
ax[0].set_title(r'$\overline{v}_y(x,t)$', pad=10)
ax[0].set_xlim([X[0], X[-1]])
ax[0].set_ylim([t[0], t[::dtplot][-1]])

ax[1].set_xlabel('$x$')
ax[1].set_ylabel('$t$')
qd_nr = ax[1].pcolormesh(xm[::dtplot,:], tm[::dtplot,:], nbar[::dtplot,:]-np.mean(nbar[-20000:], axis=0)[None,:]*0, vmin=-0.5*np.max(abs(nbar)), vmax=0.5*np.max(abs(nbar)), cmap='seismic', rasterized=True)# vmin=-0.5*np.max(abs(nbar)), vmax=0.5*np.max(abs(nbar)), cmap='seismic')
ax[1].axvspan(0, X[ib1], alpha=0.4, color='grey')
ax[1].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
plot_hatches(ax[1], 0, 0, X[ib1], t[-1])
plot_hatches(ax[1], X[ib2+1], 0, Lx - X[ib2], t[-1])
ax[1].set_title(r'$\overline{n}(x,t)$', pad=10)
ax[1].set_xlim([X[0], X[-1]])
ax[1].set_ylim([t[0], t[::dtplot][-1]])

cbar_ax1 = fig.add_subplot(gs[1, 0])
cbar_ax2 = fig.add_subplot(gs[1, 1])
cb1=fig.colorbar(qd_vy, cax=cbar_ax1, orientation='horizontal')
cb2=fig.colorbar(qd_nr, cax=cbar_ax2, orientation='horizontal')
cb1.set_label(r'$\overline{v}_y(x,t)$', labelpad=20)
cb2.set_label(r'$n_r(x,t)$', labelpad=20)

axiS = ax[1].twinx()
axiS.set_ylim([-np.max(alpha)/1e4, 6*np.max(alpha)])
axiS.plot(X, S_n(X, X0, 0, alpha, sig_n), lw=5, color='w', alpha=0.9)
axiS.plot(X, S_n(X, X0, 0, alpha, sig_n), color='green', alpha=0.9)
# axiS.axis('off')
axiS.set_ylabel(r'$S_\alpha(x,t)$')
axiS.yaxis.label.set_color('green')
axiS.set_yticks([])

[]

## Time animation

In [42]:
dtplot = 10
dxplot = 1

tm, xm = np.meshgrid(gam * t, X, indexing='ij') #run with tred

vmin_u, vmax_u = -1.5*np.max(abs(ur)), 1.5*np.max(abs(ur))
vmin_n, vmax_n = np.min(nr)-1, np.max(nr)

fig=plt.figure(figsize=(18,10), constrained_layout=True)
fig.suptitle(rf'$C={C}$')
gs = gridspec.GridSpec(3, 2, height_ratios=[10, 10, 0.5], hspace=0.4, wspace=0.3)
ax = [fig.add_subplot(gs[1, i]) for i in range(2)]
axi = [fig.add_subplot(gs[0, i]) for i in range(2)]

ax[0].set_xlabel('$x$')
ax[0].set_ylabel('$\gamma_{max}t$')
qd_vy = ax[0].pcolormesh(xm[::dtplot,::dxplot], tm[::dtplot,::dxplot], ur[::dtplot,::dxplot], vmin=-0.5*np.max(abs(ur)), vmax=0.5*np.max(abs(ur)), cmap='seismic', rasterized=True)
plot_hatches(ax[0], 0, 0, X[ib1], t[-1])
plot_hatches(ax[0], X[ib2+1], 0, Lx - X[ib2], t[-1])
ax[0].axvspan(0, X[ib1], alpha=0.4, color='grey')
ax[0].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
ax[0].set_xlim([X[ib1], X[ib2]])
ax[0].set_ylim([tm[0,0], tm[::dtplot,0][-1]])
l00, = ax[0].plot(xm[0,:], tm[0,:], color='k', lw=4, alpha=.7)

axi[0].set_xlabel('$x$')
axi[0].set_ylabel(r'$\overline{v}_y(x,t)$')
l10, = axi[0].plot(X, ur[0, :], color='blue')
plot_hatches(axi[0], X[ib2+1], vmin_u, Lx - X[ib2], vmax_u - vmin_u)
axi[0].axvspan(0, X[ib1], alpha=0.4, color='grey')
axi[0].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
axi[0].set_xlim([X[ib1], X[ib2]])
axi[0].set_ylim([vmin_u, vmax_u])
axi[0].set_title(r'Zonal velocity profile', pad=10)
axi[0].grid()

ax[1].set_xlabel('$x$')
ax[1].set_ylabel('$\gamma_{max}t$')
qd_nr = ax[1].pcolormesh(xm[::dtplot,::dxplot], tm[::dtplot,::dxplot], nbar[::dtplot,::dxplot], vmin=-0.5*np.max(abs(nbar)), vmax=0.5*np.max(abs(nbar)), cmap='seismic', rasterized=True)# vmin=-0.5*np.max(abs(nbar)), vmax=0.5*np.max(abs(nbar)), cmap='seismic')
ax[1].axvspan(0, X[ib1], alpha=0.4, color='grey')
ax[1].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
plot_hatches(ax[1], 0, 0, X[ib1], t[-1])
plot_hatches(ax[1], X[ib2+1], 0, Lx - X[ib2], t[-1])
ax[1].set_xlim([X[ib1], X[ib2]])
ax[1].set_ylim([tm[0,0], tm[::dtplot,0][-1]])
l01, = ax[1].plot(xm[0,:], tm[0,:], color='k', lw=4, alpha=.7)

axi[1].set_xlabel('$x$')
axi[1].set_ylabel(r'$n_r(x,t)$')
l11a, = axi[1].plot(X, nr[0, :], color='blue')
l11c, = axi[1].plot(X, -kap_t[0] * (X - X[ib2]) + nr[0, ib2], color='k', ls='dashed', alpha=0.8)
plot_hatches(axi[1], 0, vmin_n, X[ib1], vmax_n - vmin_n)
plot_hatches(axi[1], X[ib2+1], vmin_n, Lx - X[ib2], vmax_n - vmin_n)
axi[1].axvspan(0, X[ib1], alpha=0.4, color='grey')
axi[1].axvspan(X[ib2+1], X[-1], alpha=0.4, color='grey')
axi[1].set_xlim([X[ib1], X[ib2]])
axi[1].set_ylim([vmin_n, vmax_n])
axi[1].set_title(r'Radial density profile', pad=10)
axi[1].grid()


axiS = axi[1].twinx()
axiS.set_ylim([-np.max(alpha)/1e4, 4*np.max(alpha)])
lS1, = axiS.plot(X, S_n(X, X0, 0, alpha, sig_n), lw=4, ls=(0, (2,1)), color='w', alpha=0.9)
lS2, = axiS.plot(X, S_n(X, X0, 0, alpha, sig_n), ls=(0, (3,1)), color='green', alpha=0.9)
# axiS.axis('off')
axiS.set_ylabel(r'$S_\alpha(x,t)$')
axiS.yaxis.label.set_color('green')
axiS.set_yticks([])
cbar_ax1 = fig.add_subplot(gs[2, 0])
cbar_ax2 = fig.add_subplot(gs[2, 1])
cb1=fig.colorbar(qd_vy, cax=cbar_ax1, orientation='horizontal')
cb2=fig.colorbar(qd_nr, cax=cbar_ax2, orientation='horizontal')
cb1.set_label(r'$\overline{v}_y(x,t)$', labelpad=20)
cb2.set_label(r'$\overline{n}(x,t)$', labelpad=20)

def update(it):
    l00.set_ydata(tm[it, :])
    l01.set_ydata(tm[it, :])
    l10.set_ydata(ur[it,:])
    l11a.set_ydata(nr[it,:])
    l11c.set_ydata(-kap_t[it] * (X - X[ib2]) + nr[it, ib2])

    return l00, l01, l10, l11a, l11c,
ani = FuncAnimation(fig, update, frames=np.arange(0,t.shape[0],dtplot), blit=True, interval=1, repeat=True)